<a href="https://colab.research.google.com/github/PrinzVike/CSC4110-G02/blob/main/G02_U9_1_packaging_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 9.1: Model packaging and web app. Owner: Ronald Kipkirui

In [ ]:
# Import libraries and mount Drive
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from google.colab import drive

drive.mount('/content/drive')

# Define paths
data_path = '/content/drive/MyDrive/CSC4110_G02/data/clean_insurance.csv'
model_path = '/content/drive/MyDrive/CSC4110_G02/models/G02_model.joblib'
os.makedirs(os.path.dirname(model_path), exist_ok=True)

# Load data
df_clean = pd.read_csv(data_path)
X = df_clean.drop(columns=['charges', 'log_charges', 'bmi_band'])
y = df_clean['log_charges']

# Split data (same as Task 3.2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create the full pipeline (StandardScaler + Ridge Model)
final_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=10))
])

# Train the pipeline on the training data
final_pipe.fit(X_train, y_train)

# Save the pipeline to the models folder
joblib.dump(final_pipe, model_path)
print("Model saved successfully to:", model_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved successfully to: /content/drive/MyDrive/CSC4110_G02/models/G02_model.joblib


In [ ]:
# Reload the model to prove it works
loaded_model = joblib.load(model_path)

# Predict one test row
test_row = X_test.iloc[[0]]
prediction = loaded_model.predict(test_row)
print(f"Prediction for the first test row (Log Charges): {prediction[0]:.4f}")

# Calculate Test RMSE to use in the web app
y_pred = loaded_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.4f}")

# Create the final_pipe variable as required by graded evidence
final_pipe = loaded_model
print("final_pipe variable created successfully!")

Prediction for the first test row (Log Charges): 9.0010
Test RMSE: 0.3875
final_pipe variable created successfully!


### Written Answer
The stages of our workflow started with data collection (the insurance dataset), followed by data cleaning (handling duplicates and outliers), exploratory data analysis (visualizing the right-skewed charges), feature engineering (creating log_charges and bmi_smoker), and data splitting. We then trained and evaluated several regression models, selecting the Ridge pipeline as the best. Finally, we packaged the winning model and deployed it as an interactive Streamlit web app.

We saved the whole pipeline (`final_pipe`) rather than just the model because the pipeline includes the `StandardScaler`. If we only saved the model, the web app would feed raw, unscaled numbers into a model that expects scaled data, resulting in wildly inaccurate predictions.